In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2


In [2]:
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "Resist", "Salin", "ORP", "IP", "Cor"] # 10 entradas
TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

N_COMPONENTS = 6

In [3]:
Dataset = pd.read_excel("../Dados/Dados.xlsx")

Datasets = []

for n in range(1, 6):
    n_data = Dataset[ Dataset["Pontos"] == f"P{n}"]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])
    
    n_data_norm = n_data
        
    Datasets.append(n_data)

# PCA


In [4]:
from sklearn.decomposition import PCA

def TransformPCA(X_train, X_test):
    pca = PCA(n_components=N_COMPONENTS)
    
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca  = pca.transform(X_test)

    print(f"Variância (%): {np.round(pca.explained_variance_ratio_ * 100, 3)}")
    print(f"Total (%): {np.round(np.sum(pca.explained_variance_ratio_) * 100, 3)}")
    
    return pca, X_train_pca, X_test_pca

# RNA

In [5]:
def PrepareData(OriginalDataset, target):
    X_orig = OriginalDataset[PREDICTORS].values
    Y_orig = OriginalDataset[target].values
    
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_orig, Y_orig, test_size=0.2, random_state=42
    )

    x_train = SCALER.fit_transform(X_train)
    x_test  = SCALER.transform(X_test)
    
    pca, x_train, x_test = TransformPCA(x_train, x_test)
    
    return pca, x_train, x_test, Y_train, Y_test

    
def PrintDim(x, y):
    print(f"Dimensão da entrada: {np.shape(x)}")
    print(f"Dimensão da saida: {np.shape(y)}")

In [6]:
from sklearn.metrics import mean_squared_error, r2_score

def TrainANN(
    x_train, y_train,
    x_test, y_test,
    n_hidden=[],
    lr=1e-3,
    l2_reg=1e-4,
    epochs=500,
    batch_size=8,
    verbose=0
):
    n_inputs = x_train.shape[1]

    # ======================
    # Modelo
    # ======================
    model = Sequential()

    model.add(
        Dense(
            n_hidden[0],
            activation="relu",
            kernel_regularizer=l2(l2_reg),
            input_shape=(n_inputs,)
        )
    )

    for units in n_hidden[1:]:
        model.add(
            Dense(
                units,
                activation="relu",
                kernel_regularizer=l2(l2_reg)
            )
        )

    model.add(Dense(1, activation="linear"))

    w0 = model.get_weights()


    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="mse"
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=30,
        restore_best_weights=True
    )

    # ======================
    # Treinamento
    # ======================
    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=verbose
    )
    wf = model.get_weights()

    # ======================
    # Predições (NORMALIZADAS)
    # ======================
    y_train_pred_norm = model.predict(x_train, verbose=0)
    y_test_pred_norm  = model.predict(x_test,  verbose=0)

    # ======================
    # DESNORMALIZAÇÃO
    # ======================
    y_train_real = OUT_SCALER.inverse_transform(y_train.reshape(-1, 1)).ravel()
    y_test_real  = OUT_SCALER.inverse_transform(y_test.reshape(-1, 1)).ravel()

    y_train_pred = OUT_SCALER.inverse_transform(y_train_pred_norm).ravel()
    y_test_pred  = OUT_SCALER.inverse_transform(y_test_pred_norm).ravel()

    # ======================
    # MÉTRICAS NO ESPAÇO FÍSICO
    # ======================
    metrics = {
    "mse_train": round(mean_squared_error(y_train_real, y_train_pred), 4),
    "mse_test":  round(mean_squared_error(y_test_real,  y_test_pred), 4),
    "r2_train":  round(r2_score(y_train_real, y_train_pred), 4),
    "r2_test":   round(r2_score(y_test_real,  y_test_pred), 4)
    } 

    return model, history, metrics, w0, wf


In [12]:
neurons = [[1], [2], [4], [6], [8], [10], [14], [16], [18], [20]]
all_metrics = []

for i, Dataset in enumerate(Datasets[2:]):
    
    for j, target in enumerate(TARGETS[1:]):
        
        print(f" → {target}")
        output_dir = f"./Dados/VirtualData/P{i+3}"
        pca, x_train, x_test, y_train, y_test = PrepareData(Dataset, target)
        
        y_train = OUT_SCALER.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_test  = OUT_SCALER.transform(y_test.reshape(-1, 1)).ravel()
        PrintDim(x_train, y_train)
        PrintDim(x_test, y_test)
        
        for neuron in neurons:
            for k in range(10):
                model, history, metrics, w0, wf = TrainANN(
                    x_train, y_train,
                    x_test, y_test,
                    n_hidden=neuron
                )

                # adiciona metadados
                metrics.update({
                    "P": i + 3,
                    "runTime": k,
                    "target": target,
                    "neurons": neuron[0],  # salva como int
                    "W0": str([w.round(4).tolist() for w in w0]),
                    "Wf": str([w.round(4).tolist() for w in wf]),
                })

                # cria DataFrame da linha atual
                df_new = pd.DataFrame([metrics])

                excel_file = "Results-no-vs.xlsx"
                # salva incrementalmente no Excel
                if os.path.exists(excel_file):
                    df_old = pd.read_excel(excel_file)
                    df_final = pd.concat([df_old, df_new], ignore_index=True)
                else:
                    df_final = df_new

                df_final.to_excel(excel_file, index=False)

                print(f"Modelo P{i+3}_{target}_{neuron}_{k} treinado e salvo no Excel")
    break

 → Al
Variância (%): [41.767 18.899 14.308  9.311  7.126  4.431]
Total (%): 95.843
Dimensão da entrada: (20, 6)
Dimensão da saida: (20,)
Dimensão da entrada: (5, 6)
Dimensão da saida: (5,)
Modelo P3_Al_[1]_0 treinado e salvo no Excel
Modelo P3_Al_[1]_1 treinado e salvo no Excel
Modelo P3_Al_[1]_2 treinado e salvo no Excel
Modelo P3_Al_[1]_3 treinado e salvo no Excel
Modelo P3_Al_[1]_4 treinado e salvo no Excel
Modelo P3_Al_[1]_5 treinado e salvo no Excel
Modelo P3_Al_[1]_6 treinado e salvo no Excel
Modelo P3_Al_[1]_7 treinado e salvo no Excel
Modelo P3_Al_[1]_8 treinado e salvo no Excel
Modelo P3_Al_[1]_9 treinado e salvo no Excel
Modelo P3_Al_[2]_0 treinado e salvo no Excel
Modelo P3_Al_[2]_1 treinado e salvo no Excel
Modelo P3_Al_[2]_2 treinado e salvo no Excel
Modelo P3_Al_[2]_3 treinado e salvo no Excel
Modelo P3_Al_[2]_4 treinado e salvo no Excel
Modelo P3_Al_[2]_5 treinado e salvo no Excel
Modelo P3_Al_[2]_6 treinado e salvo no Excel
Modelo P3_Al_[2]_7 treinado e salvo no Excel
M

In [ ]:
neurons = [[1], [2], [4], [6], [8], [10], [14], [16], [18], [20]]
all_metrics = []

i=3
for Dataset in Datasets[3:]:
    for j, target in enumerate(TARGETS):
        
        print(f" → {target}")
        output_dir = f"./Dados/VirtualData/P{i+1}"
        pca, x_train, x_test, y_train, y_test = PrepareData(Dataset, target)
        
        y_train = OUT_SCALER.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_test  = OUT_SCALER.transform(y_test.reshape(-1, 1)).ravel()
        PrintDim(x_train, y_train)
        PrintDim(x_test, y_test)
        
        for neuron in neurons:
            for k in range(10):
                model, history, metrics, w0, wf = TrainANN(
                    x_train, y_train,
                    x_test, y_test,
                    n_hidden=neuron
                )

                # adiciona metadados
                metrics.update({
                    "P": i+1,
                    "runTime": k,
                    "target": target,
                    "neurons": neuron[0],  # salva como int
                    "W0": str([w.round(4).tolist() for w in w0]),
                    "Wf": str([w.round(4).tolist() for w in wf]),
                })

                # cria DataFrame da linha atual
                df_new = pd.DataFrame([metrics])

                excel_file = "Results-no-vs.xlsx"
                # salva incrementalmente no Excel
                if os.path.exists(excel_file):
                    df_old = pd.read_excel(excel_file)
                    df_final = pd.concat([df_old, df_new], ignore_index=True)
                else:
                    df_final = df_new

                df_final.to_excel(excel_file, index=False)

                print(f"Modelo P{i+1}_{target}_{neuron}_{k} treinado e salvo no Excel")
    i = i + 1

 → Fe
Variância (%): [41.767 18.899 14.308  9.311  7.126  4.431]
Total (%): 95.843
Dimensão da entrada: (20, 6)
Dimensão da saida: (20,)
Dimensão da entrada: (5, 6)
Dimensão da saida: (5,)
Modelo P3_Fe_[1]_0 treinado e salvo no Excel
Modelo P3_Fe_[1]_1 treinado e salvo no Excel
Modelo P3_Fe_[1]_2 treinado e salvo no Excel
Modelo P3_Fe_[1]_3 treinado e salvo no Excel
Modelo P3_Fe_[1]_4 treinado e salvo no Excel
Modelo P3_Fe_[1]_5 treinado e salvo no Excel
Modelo P3_Fe_[1]_6 treinado e salvo no Excel
Modelo P3_Fe_[1]_7 treinado e salvo no Excel
Modelo P3_Fe_[1]_8 treinado e salvo no Excel
Modelo P3_Fe_[1]_9 treinado e salvo no Excel
Modelo P3_Fe_[2]_0 treinado e salvo no Excel
Modelo P3_Fe_[2]_1 treinado e salvo no Excel
Modelo P3_Fe_[2]_2 treinado e salvo no Excel
Modelo P3_Fe_[2]_3 treinado e salvo no Excel
Modelo P3_Fe_[2]_4 treinado e salvo no Excel
Modelo P3_Fe_[2]_5 treinado e salvo no Excel
Modelo P3_Fe_[2]_6 treinado e salvo no Excel
Modelo P3_Fe_[2]_7 treinado e salvo no Excel
M

KeyboardInterrupt: 